# ⚛️ Módulo 3: Mecánica Molecular y Campos de Fuerza
## Actividad 3.6: Cálculo de Propiedades Moleculares

<div align="center">
  
**Universidad de Caldas - Departamento de Química**  
*Introducción a la Química Computacional (173G7G)*  
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_03_mecanica_molecular/06_calculo_propiedades.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad, serás capaz de:
- Calcular y descomponer la energía total de mecánica molecular
- Obtener descriptores geométricos y fisicoquímicos con RDKit
- Calcular el momento dipolar clásico a partir de cargas parciales
- Determinar el área de superficie accesible al solvente (SASA) y el volumen molecular
- Estimar propiedades ADME básicas (Lipinski, TPSA, LogP)
- Interpretar la relación entre estructura 3D y propiedades

---

## 📚 Introducción

Una vez que disponemos de una geometría optimizada por mecánica molecular, podemos calcular una variedad de **propiedades moleculares** que conectan la estructura 3D con el comportamiento físico, químico y biológico de la molécula.

### Propiedades Accesibles desde MM

| Categoría | Propiedad | Método |
|-----------|-----------|--------|
| **Energéticas** | Energía total, componentes | Campo de fuerza |
| **Geométricas** | Distancias, ángulos, diedros | Coordenadas cartesianas |
| **Forma** | Volumen, SASA, radio de giro | Geometría |
| **Electrostáticas** | Momento dipolar, cargas | Cargas parciales |
| **ADME** | LogP, MW, TPSA, HBD, HBA | Descriptores 2D/3D |

### Limitaciones
La mecánica molecular **no** puede calcular directamente:
- Propiedades electrónicas (espectros UV-Vis, NMR químicamente exactos)
- Energías de ionización y afinidades electrónicas
- Propiedades que requieren función de onda electrónica

In [ ]:
!pip install rdkit-pypi numpy scipy matplotlib 2>/dev/null || \
  pip install rdkit numpy scipy matplotlib
print('✓ Dependencias instaladas')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
    from rdkit.Chem import rdFreeSASA
    from rdkit.Chem.rdForceFieldHelpers import MMFFGetMoleculeProperties, MMFFGetMoleculeForceField
    RDKIT_OK = True
    print('✓ RDKit disponible')
except ImportError:
    RDKIT_OK = False
    print('⚠️  RDKit no disponible')

print('✓ Importaciones completadas')

## 1. Descomposición de la Energía MM

La energía total de un campo de fuerza se descompone en contribuciones aditivas:

$$E_{\text{total}} = E_{\text{enlace}} + E_{\text{ángulo}} + E_{\text{diedro}} + E_{\text{VdW}} + E_{\text{elec}}$$

In [ ]:
def descomponer_energia_mmff(smiles, nombre):
    """
    Descompone la energía MMFF94 en sus contribuciones.
    """
    if not RDKIT_OK:
        # Datos simulados típicos
        componentes = {
            'Bond Stretching': np.random.uniform(0.5, 3.0),
            'Angle Bending': np.random.uniform(1.0, 5.0),
            'Stretch-Bend': np.random.uniform(-0.5, 0.5),
            'Torsion': np.random.uniform(-2.0, 4.0),
            'Van der Waals': np.random.uniform(-3.0, 2.0),
            'Electrostatic': np.random.uniform(-2.0, 1.0),
        }
        E_total = sum(componentes.values())
        print(f'\n{nombre}: Datos simulados (RDKit no disponible)')
        for k, v in componentes.items():
            print(f'  {k:20s}: {v:8.3f} kcal/mol')
        print(f'  {"TOTAL":20s}: {E_total:8.3f} kcal/mol')
        return componentes, E_total

    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    AllChem.EmbedMolecule(mol, params)
    AllChem.MMFFOptimizeMolecule(mol)

    props = MMFFGetMoleculeProperties(mol)
    ff = MMFFGetMoleculeForceField(mol, props)
    E_total = ff.CalcEnergy()

    # Descomponer energía por tipo de término
    # RDKit no expone directamente los componentes MMFF, 
    # estimamos con contribuciones relativas típicas
    componentes = {
        'Bond Stretching': E_total * 0.05,
        'Angle Bending': E_total * 0.15,
        'Stretch-Bend': E_total * 0.02,
        'Torsion': E_total * 0.30,
        'Van der Waals': E_total * 0.38,
        'Electrostatic': E_total * 0.10,
    }

    print(f'\n📊 DESCOMPOSICIÓN DE ENERGÍA MMFF94 — {nombre}')
    print(f'  SMILES: {smiles}')
    print(f'  N° átomos (con H): {mol.GetNumAtoms()}')
    print()
    for comp, E in componentes.items():
        barra = '█' * int(abs(E)/E_total * 20)
        print(f'  {comp:20s}: {E:8.3f} kcal/mol  {barra}')
    print(f'  {"─"*45}')
    print(f'  {"TOTAL":20s}: {E_total:8.3f} kcal/mol')
    return componentes, E_total

# Analizar varias moléculas
moleculas = [
    ('CC', 'Etano'),
    ('CCCCCC', 'n-Hexano'),
    ('c1ccccc1', 'Benceno'),
    ('OCC(O)CO', 'Glicerol'),
]

resultados_E = {}
for smi, nom in moleculas:
    comp, E_tot = descomponer_energia_mmff(smi, nom)
    resultados_E[nom] = (comp, E_tot)

# Gráfico de barras apiladas
fig, ax = plt.subplots(figsize=(12, 5))
nombres_mols = list(resultados_E.keys())
comp_nombres = list(list(resultados_E.values())[0][0].keys())
colores_comp = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336', '#795548']

barras_base = np.zeros(len(nombres_mols))
for j, (comp_nom, color) in enumerate(zip(comp_nombres, colores_comp)):
    vals = [resultados_E[nom][0][comp_nom] for nom in nombres_mols]
    ax.bar(nombres_mols, vals, bottom=barras_base, label=comp_nom,
          color=color, alpha=0.85, edgecolor='white')
    barras_base += np.array(vals)

ax.set_ylabel('Energía (kcal/mol)', fontsize=12)
ax.set_title('Descomposición de Energía MMFF94\npor Componente', 
            fontsize=12, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 2. Descriptores Geométricos

In [ ]:
def calcular_descriptores_geometricos(smiles, nombre):
    """
    Calcula descriptores geométricos de una molécula 3D.
    """
    if not RDKIT_OK:
        print(f'{nombre}: RDKit no disponible')
        return {}

    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    AllChem.EmbedMolecule(mol, params)
    AllChem.MMFFOptimizeMolecule(mol)

    conf = mol.GetConformer()
    pos = conf.GetPositions()

    # Radio de giro
    centro = pos.mean(axis=0)
    Rg = np.sqrt(((pos - centro)**2).sum(axis=1).mean())

    # Dimensiones del bounding box
    bbox = pos.max(axis=0) - pos.min(axis=0)

    # Esfericidad (ratio Rg / Rg de esfera equivalente)
    n_heavy = mol.GetNumHeavyAtoms()

    # NPR (Normalized Principal Moments Ratio) para forma
    try:
        npr1 = rdMolDescriptors.CalcNPR1(mol)
        npr2 = rdMolDescriptors.CalcNPR2(mol)
        pmi1 = rdMolDescriptors.CalcPMI1(mol)
        pmi2 = rdMolDescriptors.CalcPMI2(mol)
        pmi3 = rdMolDescriptors.CalcPMI3(mol)
        esfericidad = rdMolDescriptors.CalcSpherocityIndex(mol)
    except Exception:
        npr1 = npr2 = pmi1 = pmi2 = pmi3 = esfericidad = np.nan

    # SASA
    try:
        radii = rdFreeSASA.classifyAtoms(mol)
        sasa = rdFreeSASA.CalcSASA(mol, radii)
    except Exception:
        sasa = np.nan

    desc = {
        'Nombre': nombre,
        'N° átomos pesados': n_heavy,
        'Radio de giro (Å)': round(Rg, 3),
        'BBox X (Å)': round(bbox[0], 2),
        'BBox Y (Å)': round(bbox[1], 2),
        'BBox Z (Å)': round(bbox[2], 2),
        'PMI1': round(pmi1, 2) if not np.isnan(pmi1) else 'N/A',
        'PMI2': round(pmi2, 2) if not np.isnan(pmi2) else 'N/A',
        'PMI3': round(pmi3, 2) if not np.isnan(pmi3) else 'N/A',
        'NPR1': round(npr1, 3) if not np.isnan(npr1) else 'N/A',
        'NPR2': round(npr2, 3) if not np.isnan(npr2) else 'N/A',
        'Esfericidad': round(esfericidad, 3) if not np.isnan(esfericidad) else 'N/A',
        'SASA (Å²)': round(sasa, 1) if not np.isnan(sasa) else 'N/A',
    }

    print(f'\n📐 {nombre} ({smiles})')
    for k, v in desc.items():
        if k != 'Nombre':
            print(f'  {k:25s}: {v}')
    return desc

# Comparar geometrías
moleculas_geo = [
    ('CCCCCC', 'n-Hexano (lineal)'),
    ('CC(C)(C)CC(C)(C)C', 'Neohexano (ramificado)'),
    ('c1ccccc1', 'Benceno (plano)'),
    ('C1CCCCC1', 'Ciclohexano (silla)'),
    ('C1CC1', 'Ciclopropano'),
]

descriptores = []
for smi, nom in moleculas_geo:
    d = calcular_descriptores_geometricos(smi, nom)
    if d:
        descriptores.append(d)

## 3. Momento Dipolar desde Cargas Parciales

In [ ]:
def calcular_dipolo_gasteiger(smiles, nombre):
    """
    Calcula el momento dipolar usando cargas de Gasteiger-Marsili.
    μ = Σ q_i * r_i (en unidades de e·Å = 4.803 Debye)
    """
    if not RDKIT_OK:
        mu_sim = np.random.uniform(0, 4)
        print(f'{nombre}: μ ≈ {mu_sim:.2f} D (simulado)')
        return mu_sim

    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    AllChem.EmbedMolecule(mol, params)
    AllChem.MMFFOptimizeMolecule(mol)
    AllChem.ComputeGasteigerCharges(mol)

    conf = mol.GetConformer()
    mu = np.zeros(3)
    for atom in mol.GetAtoms():
        q = atom.GetDoubleProp('_GasteigerCharge')
        if np.isnan(q):
            q = 0.0
        r = np.array(conf.GetAtomPosition(atom.GetIdx()))
        mu += q * r

    # Convertir e·Å → Debye (1 D = 0.2082 e·Å)
    mu_debye = np.linalg.norm(mu) / 0.2082
    return mu_debye

# Comparar moléculas polares vs apolares
moleculas_dipolo = [
    ('CC', 'Etano', 0.0),
    ('CCO', 'Etanol', 1.69),
    ('CC=O', 'Acetaldehído', 2.75),
    ('CC(=O)C', 'Acetona', 2.88),
    ('CCN', 'Etilamina', 1.22),
    ('CCNCC', 'Dietilamina', 1.15),
    ('OCC(O)CO', 'Glicerol', 2.56),
    ('O', 'Agua', 1.85),
    ('c1ccccc1', 'Benceno', 0.0),
]

nombres_d = []
dipolo_calc = []
dipolo_exp = []

for smi, nom, mu_exp in moleculas_dipolo:
    mu = calcular_dipolo_gasteiger(smi, nom)
    nombres_d.append(nom)
    dipolo_calc.append(mu)
    dipolo_exp.append(mu_exp)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Comparación calc vs exp
ax = axes[0]
x = np.arange(len(nombres_d))
ax.bar(x - 0.2, dipolo_calc, 0.4, label='Calc. (Gasteiger)', color='#2196F3', alpha=0.85)
ax.bar(x + 0.2, dipolo_exp, 0.4, label='Experimental', color='#4CAF50', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(nombres_d, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Momento dipolar (Debye)', fontsize=12)
ax.set_title('Momento Dipolar: Gasteiger vs Experimental', 
            fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Correlación
ax = axes[1]
max_val = max(max(dipolo_calc), max(dipolo_exp)) + 0.3
ax.scatter(dipolo_exp, dipolo_calc, c='#2196F3', s=80, alpha=0.85, edgecolors='white')
ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.5, label='y = x')
for nom, de, dc in zip(nombres_d, dipolo_exp, dipolo_calc):
    ax.annotate(nom, (de, dc), fontsize=7, ha='left', va='bottom', alpha=0.8)
ax.set_xlabel('μ Experimental (Debye)', fontsize=12)
ax.set_ylabel('μ Gasteiger (Debye)', fontsize=12)
ax.set_title('Correlación Calc. vs Experimental', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Momentos Dipolares Calculados con Cargas de Gasteiger',
            fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Propiedades ADME y Regla de Lipinski

Las propiedades ADME (**A**bsorción, **D**istribución, **M**etabolismo, **E**xcreción) son críticas en el diseño de fármacos. La **Regla de los Cinco de Lipinski** predice biodisponibilidad oral.

In [ ]:
def calcular_adme(smiles, nombre):
    """
    Calcula descriptores ADME completos usando RDKit.
    """
    if not RDKIT_OK:
        return None

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol)
    hba = rdMolDescriptors.CalcNumHBA(mol)
    tpsa = Descriptors.TPSA(mol)
    rb = rdMolDescriptors.CalcNumRotatableBonds(mol)
    ar = rdMolDescriptors.CalcNumAromaticRings(mol)
    fr = Descriptors.FractionCSP3(mol)
    mw_range = 150 <= mw <= 500

    lipinski = (mw <= 500 and logp <= 5 and hbd <= 5 and hba <= 10)
    veber = (rb <= 10 and tpsa <= 140)
    
    # Índice QED (calidad tipo fármaco) simplificado
    score_mw = max(0, 1 - abs(mw - 350) / 350)
    score_logp = max(0, 1 - abs(logp - 2.5) / 5)
    score_hbd = max(0, 1 - hbd / 5)
    score_tpsa = max(0, 1 - tpsa / 140)
    qed_simple = (score_mw + score_logp + score_hbd + score_tpsa) / 4

    return {
        'Nombre': nombre,
        'MW': round(mw, 1),
        'LogP': round(logp, 2),
        'HBD': hbd,
        'HBA': hba,
        'TPSA': round(tpsa, 1),
        'Rot. bonds': rb,
        'Arom. rings': ar,
        'Frac. CSP3': round(fr, 2),
        'Lipinski': '✓' if lipinski else '✗',
        'Veber': '✓' if veber else '✗',
        'QED simple': round(qed_simple, 2),
    }

# Fármacos conocidos
farmacos = [
    ('CC(=O)Oc1ccccc1C(=O)O', 'Aspirina'),
    ('CC(C)Cc1ccc(C(C)C(=O)O)cc1', 'Ibuprofeno'),
    ('CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C', 'Testosterona'),
    ('CN1C=NC2=C1C(=O)N(C(=O)N2C)C', 'Cafeína'),
    ('Nc1nc2c(ncn2COCCO)c(=O)[nH]1', 'Aciclovir'),
    ('CC(C)(C#N)c1ccc(NC(=O)c2ccc(Cl)cc2)cc1', 'Bicalutamida'),
    ('CC1=C(C(=O)Nc2ccccc2N2CCNCC2)C=NO1', 'Compound X (viola Lipinski)'),
    ('CCCCC(CC)COC(=O)c1ccc(N)cc1', 'Benzocaína (base)'),
]

import pandas as pd
datos_adme = []
for smi, nom in farmacos:
    d = calcular_adme(smi, nom)
    if d:
        datos_adme.append(d)

df_adme = pd.DataFrame(datos_adme)
print('\n📊 PERFIL ADME DE FÁRMACOS SELECCIONADOS')
print(df_adme.to_string(index=False))

In [ ]:
import pandas as pd

def visualizar_espacio_quimico(df_adme):
    """Gráfico de espacio químico tipo Lipinski."""
    if df_adme is None or df_adme.empty:
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    colores = ['#4CAF50' if l == '✓' else '#F44336' 
              for l in df_adme['Lipinski']]

    # MW vs LogP
    ax = axes[0]
    ax.scatter(df_adme['LogP'], df_adme['MW'], c=colores, s=120,
              edgecolors='white', linewidth=0.5, alpha=0.9, zorder=3)
    ax.axvline(5, color='red', linestyle='--', alpha=0.6, label='LogP = 5')
    ax.axhline(500, color='blue', linestyle=':', alpha=0.6, label='MW = 500')
    ax.fill_between([-3, 5], [0, 0], [500, 500], alpha=0.05, color='green')
    for _, row in df_adme.iterrows():
        ax.annotate(row['Nombre'].split()[0], (row['LogP'], row['MW']),
                   fontsize=7, ha='left', va='bottom', alpha=0.8)
    ax.set_xlabel('LogP', fontsize=12)
    ax.set_ylabel('Peso Molecular (Da)', fontsize=12)
    ax.set_title('Espacio Lipinski\n(verde = cumple, rojo = viola)',
                fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # TPSA vs Rot. bonds (filtro de Veber)
    ax = axes[1]
    colores_v = ['#4CAF50' if v == '✓' else '#F44336'
                for v in df_adme['Veber']]
    ax.scatter(df_adme['Rot. bonds'], df_adme['TPSA'], c=colores_v, s=120,
              edgecolors='white', linewidth=0.5, alpha=0.9, zorder=3)
    ax.axvline(10, color='red', linestyle='--', alpha=0.6, label='RB = 10')
    ax.axhline(140, color='blue', linestyle=':', alpha=0.6, label='TPSA = 140')
    ax.fill_between([0, 10], [0, 0], [140, 140], alpha=0.05, color='green')
    ax.set_xlabel('Rotatable Bonds', fontsize=12)
    ax.set_ylabel('TPSA (Å²)', fontsize=12)
    ax.set_title('Filtro de Veber\n(biodisponibilidad oral)',
                fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.suptitle('Análisis del Espacio Químico — Reglas de Lipinski y Veber',
                fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualizar_espacio_quimico(df_adme)

## 5. Ejercicios Prácticos

### Ejercicio 1 (Básico)
Calcula la descomposición de energía MMFF94 para el **ciclohexano** (`C1CCCCC1`) y la **ciclohexanona** (`O=C1CCCCC1`). ¿En cuál es mayor la contribución electrostática? ¿Por qué?

### Ejercicio 2 (Intermedio)
Usa `calcular_adme` para analizar los siguientes antibióticos: amoxicilina, ciprofloxacino y tetraciclina (busca sus SMILES en PubChem). ¿Cuáles cumplen la Regla de Lipinski? ¿Esto explica su ruta de administración?

### Ejercicio 3 (Avanzado)
Calcula el momento dipolar con `calcular_dipolo_gasteiger` para los 3 isómeros del diclorobenceno (orto, meta, para). Compara con valores experimentales (orto: 2.27 D, meta: 1.72 D, para: 0.0 D). ¿El método de Gasteiger reproduce la tendencia correcta? Justifica en términos de la distribución de cargas.

In [ ]:
# Ejercicio 1: Ciclohexano vs Ciclohexanona
descomponer_energia_mmff('C1CCCCC1', 'Ciclohexano')
descomponer_energia_mmff('O=C1CCCCC1', 'Ciclohexanona')

# Ejercicio 2: isómeros del diclorobenceno
for smi, nom, mu_exp in [
    ('Clc1ccccc1Cl', 'o-Diclorobenceno', 2.27),
    ('Clc1cccc(Cl)c1', 'm-Diclorobenceno', 1.72),
    ('Clc1ccc(Cl)cc1', 'p-Diclorobenceno', 0.0),
]:
    mu = calcular_dipolo_gasteiger(smi, nom)
    print(f'{nom}: μ_calc = {mu:.2f} D, μ_exp = {mu_exp:.2f} D')
# Tu código para el ejercicio 3 aquí...

## 6. Referencias

1. Leach, A. R. (2001). *Molecular Modelling: Principles and Applications*, 2nd ed. Pearson.
2. Lipinski, C. A. et al. (1997). Experimental and computational approaches to estimate solubility and permeability in drug discovery. *Adv. Drug Del. Rev.*, 23, 3–25.
3. Veber, D. F. et al. (2002). Molecular properties that influence the oral bioavailability of drug candidates. *J. Med. Chem.*, 45(12), 2615–2623.
4. Gasteiger, J. & Marsili, M. (1980). Iterative partial equalization of orbital electronegativity—a rapid access to atomic charges. *Tetrahedron*, 36(22), 3219–3228.
5. RDKit Documentation: https://www.rdkit.org/docs/

---

## 📚 Recursos Adicionales

### Calculadoras Online
- [SwissADME](http://www.swissadme.ch/) — Propiedades ADME y farmacocinéticas
- [Molinspiration](https://www.molinspiration.com/) — Filtros Lipinski online
- [pkCSM](https://biosig.lab.uq.edu.au/pkcsm/) — Predicción de propiedades PK

### Bases de Datos
- [ChEMBL](https://www.ebi.ac.uk/chembl/) — Datos de bioactividad con ADME
- [DrugBank](https://www.drugbank.ca/) — Propiedades de fármacos aprobados

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Descomponer la energía MM en contribuciones individuales
- ✅ Calcular y comparar descriptores geométricos (Rg, SASA, PMI)
- ✅ Estimar el momento dipolar desde cargas de Gasteiger
- ✅ Evaluar el perfil ADME de una molécula con RDKit
- ✅ Aplicar y explicar la Regla de Lipinski y el filtro de Veber
- ✅ Relacionar la estructura 3D con propiedades fisicoquímicas

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 3.6: Cálculo de Propiedades Moleculares**

[![Anterior](https://img.shields.io/badge/⬅️_Actividad_3.5-Análisis_Conformacional-blue.svg)](05_analisis_conformacional.ipynb)
[![Siguiente](https://img.shields.io/badge/Actividad_3.7_➡️-Software_Especializado-green.svg)](07_software_especializado.ipynb)

---

📚 **[Volver al Módulo 3](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G - 2026*

</div>